# Needle in a Haystack — Long Context Retrieval

**Community Benchmark** | Adaptation of the classic NIAH test (Kamradt 2023)

A specific fact (the "needle") is hidden at various positions within a long document (the "haystack"). The model must retrieve the fact.

**Design:** Vary needle position (beginning, 25%, 50%, 75%, end) and haystack size (1K, 5K, 10K, 20K words).

**Metric:** Retrieval accuracy by position × context length.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()

In [ ]:
# Haystack: fictional academic text paragraphs (~100 words each)
HAYSTACK_PARAGRAPHS = [
    "The mineral composition of the Valderi Basin sediments reveals a complex geological history spanning approximately 340 million years. Stratigraphic analysis conducted by the Pangaea Institute identified fourteen distinct depositional layers, each corresponding to a major climatic shift in the late Paleozoic era. The predominance of feldspar in the upper strata suggests extensive weathering of granitic source rocks during periods of elevated precipitation. Meanwhile, the lower carbonate layers indicate prolonged marine incursion across what is now a continental interior.",
    "Recent advances in computational fluid dynamics have enabled more accurate modeling of turbulent flow patterns in industrial heat exchangers. The Reynolds-averaged Navier-Stokes equations, when coupled with the k-epsilon turbulence model, provide predictions within 3.7 percent of experimental measurements for fully developed pipe flow. However, transition regimes between laminar and turbulent states continue to pose challenges, particularly in geometries involving sharp contractions followed by sudden expansions.",
    "The economic impact of autonomous shipping on Mediterranean trade routes has been the subject of extensive analysis since the first unmanned cargo vessels completed transits in 2024. Port authorities in Genoa, Piraeus, and Valencia have invested a combined 2.8 billion euros in automated berthing infrastructure. Labor unions representing dockworkers have negotiated transition agreements that guarantee retraining programs for displaced personnel over a five-year period.",
    "Anthropological studies of the Karimu people of central Papua New Guinea document a sophisticated system of ecological knowledge accumulated over approximately 8,000 years of continuous habitation. Their classification of plant species distinguishes 847 named varieties, exceeding the taxonomic resolution of Western botanical surveys conducted in the same region. Ethnobotanist Dr. Hiroshi Yamada's comparative analysis suggests that indigenous categorization systems prioritize functional and medicinal properties over morphological similarities.",
    "The development of metamaterials with negative refractive indices has opened new possibilities in electromagnetic cloaking and super-resolution imaging. Fabrication techniques using electron beam lithography can now produce structures with feature sizes below 40 nanometers, enabling operation at near-infrared wavelengths. The primary limitation remains the narrow bandwidth over which negative refraction is maintained, typically spanning less than 50 nanometers in current implementations.",
    "Historical analysis of voting patterns in the Swiss canton system reveals remarkably stable political alignments spanning more than two centuries. The introduction of proportional representation in 1919 reduced the dominance of liberal parties in urban cantons but had minimal effect on conservative strongholds in the alpine regions. Referendum participation rates have shown a steady decline from 78 percent in 1950 to approximately 43 percent in recent decades.",
    "Marine biologists studying coral reef resilience in the Maldivian archipelago have identified genetic markers associated with thermal tolerance in Acropora digitifera colonies. Specimens collected from shallow lagoons where temperatures regularly exceed 33 degrees Celsius exhibit upregulation of heat shock protein genes at levels three to five times higher than conspecifics from deeper reef slopes. This natural variation offers potential pathways for assisted gene flow conservation strategies.",
    "The optimization of battery cathode materials for next-generation electric vehicles has focused on nickel-manganese-cobalt layered oxides with increasing nickel content. Compositions approaching 90 percent nickel offer theoretical energy densities of 800 watt-hours per kilogram but suffer from structural instability during deep discharge cycles. Surface coating with aluminum oxide at thicknesses of 2 to 5 nanometers has shown promising results in extending cycle life beyond 1,500 charge-discharge cycles.",
    "Archaeological excavations at the Knossos Administrative Complex have yielded a trove of Linear A tablets that may finally provide sufficient material for decipherment. The 2024 season recovered 347 tablets from a sealed archive room, more than doubling the existing corpus. Computational linguistic analysis using transformer-based language models has identified recurring syntactic patterns consistent with an agglutinative morphological structure.",
    "The gravitational wave observatory network, now comprising twelve operational detectors on six continents, has catalogued over 400 binary merger events since the first detection in 2015. Statistical analysis of mass distributions reveals a gap between 3 and 5 solar masses where neither neutron stars nor black holes appear to form, confirming theoretical predictions about supernova core collapse dynamics.",
    "Precision agriculture techniques employing hyperspectral satellite imagery can detect nitrogen deficiency in cereal crops up to two weeks before visible symptoms appear. The spectral signature between 700 and 740 nanometers, known as the red edge, shifts toward shorter wavelengths as chlorophyll content decreases. Commercial implementation by AgriSat Corp has demonstrated yield improvements of 8 to 12 percent in wheat fields across the Canadian prairies.",
    "The neuroscience of decision-making under uncertainty has been significantly advanced by optogenetic studies in murine models. Selective activation of dopaminergic neurons in the ventral tegmental area during a probabilistic reward task shifts subjects toward risk-seeking behavior, while inhibition of the same population promotes conservative choice strategies. These findings support the hypothesis that dopamine encodes a prediction error signal rather than reward value per se.",
    "Forensic accounting investigations of cryptocurrency exchanges have revealed systematic patterns of wash trading that inflate reported volumes by factors of 10 to 50. The Benford's law analysis applied to transaction ledgers of 23 major exchanges identified statistically significant deviations in first-digit distributions, consistent with fabricated data. Regulatory authorities in Singapore and Switzerland have subsequently implemented real-time monitoring requirements.",
    "The dynamics of supercooled water near its glass transition temperature continue to challenge theoretical frameworks in condensed matter physics. Neutron scattering experiments at temperatures below 150 kelvin reveal a secondary relaxation process with an activation energy of approximately 0.5 electron volts, suggesting cooperative molecular rearrangements involving clusters of 10 to 15 water molecules.",
    "Urban planning research in Copenhagen has demonstrated that conversion of automobile lanes to protected bicycle infrastructure increases retail revenue along affected corridors by 20 to 35 percent. The mechanism appears to operate through increased pedestrian traffic and longer dwelling times, as cyclists stop more frequently than motorists. Economic modeling by the Danish Transport Authority projects a benefit-cost ratio of 4.7 for the city's 2030 bicycle network expansion plan.",
    "The International Seed Vault in Svalbard now houses over 1.2 million seed samples representing more than 6,000 plant species from every country with active agricultural research. The facility maintains a constant temperature of minus 18 degrees Celsius using natural permafrost supplemented by mechanical cooling. Recent concerns about Arctic warming prompted the installation of a secondary cooling system with redundant power from both hydroelectric and diesel generators.",
    "Quantum error correction codes based on the surface code architecture require a physical-to-logical qubit ratio of approximately 1,000 to 1 for fault-tolerant computation. Current prototype processors from IBM and Google have demonstrated error rates approaching the threshold of 0.1 percent needed to make surface codes practical. The roadmap toward useful quantum advantage in drug discovery and materials science envisions machines with 10,000 to 100,000 physical qubits by 2030.",
    "Paleontological discoveries in the Kem Kem beds of Morocco have revealed an unprecedented diversity of large predatory dinosaurs coexisting in a single ecosystem during the mid-Cretaceous period. At least seven species of theropod exceeding 6 meters in length have been identified, including spinosaurids, carcharodontosaurids, and abelisaurids. This apparent violation of competitive exclusion principles may be explained by niche partitioning across aquatic and terrestrial prey resources.",
    "The pharmacokinetics of mRNA-based therapeutics depend critically on the lipid nanoparticle delivery system used to protect the payload from enzymatic degradation. Ionizable lipids with pKa values between 6.2 and 6.5 achieve optimal endosomal escape while minimizing off-target inflammatory responses. Manufacturing scale-up from laboratory to commercial production volumes introduces challenges in maintaining nanoparticle size distributions within the 80 to 100 nanometer range required for liver targeting.",
    "Climate-driven migration of boreal forest species northward at rates of 30 to 50 kilometers per decade has created novel ecological communities with no historical precedent. The interaction between advancing spruce and retreating tundra vegetation generates feedback loops affecting surface albedo, evapotranspiration, and permafrost stability. Earth system models incorporating dynamic vegetation modules predict that the boreal treeline will shift 200 to 500 kilometers poleward by 2100.",
]

# Needles: distinctive facts that are easy to verify
NEEDLES = [
    {
        "fact": "The Castellini Prize for computational linguistics was awarded to Dr. Priya Ramanathan of the University of Melbourne in 2024 for her work on cross-lingual semantic alignment.",
        "question": "Who won the Castellini Prize for computational linguistics, and from which university?",
        "answer": "Dr. Priya Ramanathan from the University of Melbourne",
    },
    {
        "fact": "The deepest point in Lake Vostok, beneath 3,769 meters of Antarctic ice, was measured at 1,067 meters during the 2024 sonar survey by the Polar Research Consortium.",
        "question": "What is the deepest point in Lake Vostok and who measured it?",
        "answer": "1,067 meters, measured by the Polar Research Consortium",
    },
    {
        "fact": "The synthetic enzyme designated KX-7 catalyzes the breakdown of PFAS compounds at room temperature, achieving 94 percent degradation within 48 hours according to results published by the Zurich Institute of Applied Chemistry.",
        "question": "What is the degradation rate of the synthetic enzyme KX-7 on PFAS compounds?",
        "answer": "94 percent degradation within 48 hours",
    },
]

print(f"Haystack: {len(HAYSTACK_PARAGRAPHS)} paragraphs")
print(f"Needles: {len(NEEDLES)} distinct facts")

In [ ]:
random.seed(777)

# Context sizes (in paragraphs)
CONTEXT_SIZES = [5, 10, 15, 20]  # ~500, ~1000, ~1500, ~2000 words
POSITIONS = [0.0, 0.25, 0.5, 0.75, 1.0]  # beginning to end

data = []
task_id = 0

for needle in NEEDLES:
    for ctx_size in CONTEXT_SIZES:
        for pos_frac in POSITIONS:
            # Select haystack paragraphs
            selected = random.sample(HAYSTACK_PARAGRAPHS, min(ctx_size, len(HAYSTACK_PARAGRAPHS)))
            if ctx_size > len(HAYSTACK_PARAGRAPHS):
                selected = selected * (ctx_size // len(HAYSTACK_PARAGRAPHS) + 1)
                selected = selected[:ctx_size]
            
            # Insert needle at position
            insert_idx = max(0, min(int(pos_frac * len(selected)), len(selected) - 1))
            selected.insert(insert_idx, needle["fact"])
            
            document = "\n\n".join(selected)
            word_count = len(document.split())
            
            prompt = (
                "Read the following document carefully and answer the question at the end.\n\n"
                f"---\n{document}\n---\n\n"
                f"Question: {needle['question']}\n\n"
                "Answer concisely based only on information in the document above."
            )
            
            data.append({
                "task_id": task_id,
                "prompt": prompt,
                "expected": needle["answer"],
                "needle_id": NEEDLES.index(needle),
                "context_paragraphs": ctx_size,
                "context_words": word_count,
                "position": pos_frac,
                "position_label": ["beginning", "25%", "middle", "75%", "end"][POSITIONS.index(pos_frac)],
            })
            task_id += 1

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} items")
print(f"By context size: {dict(df_all['context_paragraphs'].value_counts())}")
print(f"By position: {dict(df_all['position_label'].value_counts())}")

## Task Definition

In [ ]:
@kbench.task(
    name="needle_in_haystack",
    description="Retrieve a specific fact hidden in a long document — classic NIAH test"
)
def needle_in_haystack(
    llm,
    prompt: str,
    expected: str,
) -> bool:
    response = llm.prompt(prompt)
    response = strip_thinking(response).lower()
    expected_lower = expected.lower()
    
    # Check if key parts of the expected answer appear in the response
    key_parts = [p.strip() for p in re.split(r'[,;]', expected_lower) if p.strip()]
    matches = sum(1 for part in key_parts if part in response)
    return matches >= len(key_parts) * 0.5  # At least half the key parts match

## Run Evaluation

In [ ]:
eval_df = df_all[["prompt", "expected"]].copy()

print(f"Running {len(eval_df)} items with kbench.llm...")
runs = needle_in_haystack.evaluate(
    llm=kbench.llm,
    evaluation_data=eval_df,
    n_jobs=2,
    timeout=180,
    max_attempts=2,
)
results = runs.as_dataframe()
results["correct"] = results["result"].apply(lambda r: float(r) if isinstance(r, bool) else float(r))
results = results.merge(
    df_all[["prompt", "context_paragraphs", "position", "position_label", "needle_id"]],
    on="prompt", how="left"
)
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
import matplotlib.pyplot as plt

print("=== Accuracy by Position × Context Size ===")
pivot = results.groupby(["position_label", "context_paragraphs"])["correct"].mean().unstack()
print(pivot.to_string(float_format="%.2f"))

print("\n=== Overall by Position ===")
print(results.groupby("position_label")["correct"].mean().to_string(float_format="%.3f"))

print("\n=== Overall by Context Size ===")
print(results.groupby("context_paragraphs")["correct"].mean().to_string(float_format="%.3f"))

# Heatmap
fig, ax = plt.subplots(figsize=(10, 6))
pos_order = ["beginning", "25%", "middle", "75%", "end"]
ctx_order = sorted(results["context_paragraphs"].unique())

heatmap_data = np.zeros((len(pos_order), len(ctx_order)))
for i, pos in enumerate(pos_order):
    for j, ctx in enumerate(ctx_order):
        subset = results[(results["position_label"] == pos) & (results["context_paragraphs"] == ctx)]
        heatmap_data[i, j] = subset["correct"].mean() if len(subset) > 0 else 0

im = ax.imshow(heatmap_data, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(ctx_order)))
ax.set_xticklabels([f"{c} para" for c in ctx_order])
ax.set_yticks(range(len(pos_order)))
ax.set_yticklabels(pos_order)
ax.set_xlabel("Context Size")
ax.set_ylabel("Needle Position")
ax.set_title("Needle in a Haystack: Retrieval Accuracy")

for i in range(len(pos_order)):
    for j in range(len(ctx_order)):
        ax.text(j, i, f"{heatmap_data[i,j]:.0%}", ha="center", va="center",
                color="black" if heatmap_data[i,j] > 0.5 else "white", fontsize=12)

plt.colorbar(im, label="Accuracy")
plt.tight_layout()
plt.savefig("niah_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nPlot saved to niah_results.png")
print("\nReference: Liu et al. (2024) 'Lost in the Middle' showed U-shaped performance curves.")